In [30]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import minimize

# Configuration
DATA_DIR = Path.cwd() / "testfiles_" / "data"
CSV_PATH = DATA_DIR / "test5_2.csv"

# Load covariance matrix
covar_data = pd.read_csv(CSV_PATH, header=0)
covar_matrix = covar_data.values

def calculate_portfolio_vol(w, cov):
    """Calculate portfolio volatility"""
    return np.sqrt(np.dot(w, np.dot(cov, w)))

def get_risk_contributions(w, cov):
    """Calculate risk contribution of each asset"""
    port_vol = calculate_portfolio_vol(w, cov)
    marginal_risk = np.dot(cov, w)
    contributions = w * marginal_risk / port_vol
    return contributions

def objective_with_budget(w, cov, budget_vec):
    """Minimize variance of budget-adjusted risk contributions"""
    risk_contrib = get_risk_contributions(w, cov)
    normalized_contrib = risk_contrib / budget_vec
    target_value = np.mean(normalized_contrib)
    differences = normalized_contrib - target_value
    return np.sum(differences ** 2) * 1e5

# Setup
num_assets = covar_matrix.shape[0]
starting_weights = np.ones(num_assets) / num_assets

# Risk budget: x1-x4 share 1/2 equally, x5 gets 1/2
budget_allocation = np.array([1.4, 1.4, 1.0, 1.0, 0.5])

constraint_sum = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}
bounds_positive = tuple((0, None) for _ in range(num_assets))

# Optimize
result = minimize(
    objective_with_budget,
    starting_weights,
    args=(covar_matrix, budget_allocation),
    method='SLSQP',
    bounds=bounds_positive,
    constraints=constraint_sum,
    options={'ftol': 1e-12, 'maxiter': 1000}
)

# Get final weights
optimal_weights = result.x
optimal_weights = optimal_weights / np.sum(optimal_weights)

# Output
print('w')
for w in optimal_weights:
    print(w)

w
0.06624738677783289
0.048206777937460435
0.07812794020616239
0.36791238561148437
0.43950550946705985
